In [8]:
from synthesizer import distr_concat, load_json_to_df
import json
import pandas as pd
from scipy.spatial import KDTree
from sklearn.neighbors import NearestNeighbors

In [29]:
import numpy as np

source_id      0.000000e+00
grain_start    9.600000e+03
grain_size     4.800000e+02
centroid       5.999954e+02
flatness       1.688917e-02
kurtosis       1.083947e+00
flux           1.934955e-08
energy         1.334604e-09
crest          1.184677e+00
rms            4.489678e-06
eef            0.000000e+00
eer            0.000000e+00
band_width     1.956155e+02
decrease      -4.562375e+00
entropy        0.000000e+00
spread         1.366182e+03
slope         -2.262475e-04
dtype: float64

In [64]:
with open("metadata\metro_sample_2.json", "r") as f:
    df = pd.read_json(f)
df = df.fillna(0)
cols_to_check = df.iloc[:, 3:] #remove all the "silent" rows/grains
df = df[cols_to_check.ne(0).any(axis=1)]
vector_df = df.iloc[:, 3:]
normalized_df=(vector_df-vector_df.min())/(vector_df.max()-vector_df.min())
tree = KDTree(normalized_df)
normalized_df

,centroid,flatness,kurtosis,flux,energy,crest,rms,eef,eer,band_width,decrease,entropy,spread,slope
20,0.549345,0.674666,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.984834,0.000000,0.342871,0.999999
21,1.000000,1.000000,0.013909,7.467325e-10,5.764326e-10,0.089204,0.000025,0.144578,0.654132,0.006142,0.965712,1.000000,0.709941,1.000000
22,0.670746,0.846890,0.015085,2.718034e-09,1.184461e-09,0.304155,0.000030,0.144578,0.654132,0.005431,0.856800,0.901836,0.810096,0.999973
23,0.770870,0.933253,0.010082,3.309464e-09,1.720362e-09,0.172081,0.000043,0.144578,0.654132,0.008399,0.934554,0.971510,0.757671,0.999970
24,0.409753,0.377061,0.059493,3.600651e-06,3.500907e-06,0.313580,0.001871,0.144601,0.654139,0.038211,0.920732,0.793428,0.550259,0.998030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9368,0.055416,0.026733,0.482343,2.171198e-02,1.426306e-02,0.590961,0.131170,0.201078,0.683308,0.104330,0.644149,0.554965,0.062171,0.846791
9369,0.068781,0.029482,0.427966,2.341775e-02,1.235580e-02,0.524126,0.126434,0.194392,0.678607,0.106869,0.803712,0.580153,0.073415,0.854732
9370,0.069897,0.031795,0.454414,2.633765e-02,1.438808e-02,0.477165,0.129239,0.199865,0.681753,0.107532,0.745806,0.581834,0.070716,0.846796
9371,0.078969,0.030814,0.409462,1.684907e-02,9.676584e-03,0.422131,0.115143,0.187740,0.673115,0.102931,0.852217,0.603713,0.068320,0.860775


In [63]:
seed = np.random.seed(111)
random_starting_point = np.random.rand(1, len(vector_df.iloc[0]))[0]
nn = tree.query(random_starting_point)
random_starting_point, nn

(array([0.61217018, 0.16906975, 0.43605902, 0.76926247, 0.2953253 ,
        0.14916296, 0.02247832, 0.42022449, 0.23868214, 0.33765619,
        0.99071246, 0.23772645, 0.08119266, 0.66960024]),
 (1.110117533097052, np.int64(3049)))

In [ ]:
# generate path across n-dimensional space
start = normalized_df.iloc[0]
stop = normalized_df.iloc[-1]
# start, end
trajectory = np.linspace(start,stop) #default 50 points
num_particles = 20
n_features = len(vector_df.iloc[0])
particles = trajectory[0] + np.random.normal(0, 0.1, (num_particles, n_features)) #change 0.1 for spread degree
weights = np.ones(num_particles) / num_particles

output_indices = []
total_steps = len(trajectory)
sigma = 0.1 #the smaller the value, the smaller the distance between the nearest neighbour and target grain
# sigma needs to be very high, when alpha is high
alpha = 0.5

for t in range(total_steps):
    target = trajectory[t]
    # print(target)
    particles += np.random.normal(0, alpha, particles.shape) #add noise to nearby particles 
    
    for i in range(num_particles):
        dist, grain_idx = tree.query(particles[i])  
        weights[i] = np.exp(-dist**2 / (2 * sigma**2)) #Gaussian radial basis func, dist >> prob
        # print(weights[i])
    # print(weights)
    weights /= np.sum(weights)
    # print(weights)
    
    best_particle_idx = np.argmax(weights)
    chosen_grain = normalized_df.iloc[best_particle_idx]
    # print("INDEX: ", best_particle_idx, particles[best_particle_idx])
    _, final_grain_idx = tree.query(particles[best_particle_idx])
    # print("INDEX: ", final_grain_idx, normalized_df.iloc[final_grain_idx])
    output_indices.append(final_grain_idx)
    
    indices = np.random.choice(range(num_particles), size=num_particles, p=weights)
    particles = particles[indices]

3.2736741950943295e-74
3.038223728357262e-49
2.443338572700113e-82
4.556493868147646e-35
1.3886190156867684e-104
7.220945682114825e-51
1.0118625119193844e-64
1.1010626397782215e-54
1.7364980044645316e-85
5.185674448205365e-106
3.6969326272541946e-103
5.116240343624552e-116
4.920458862620255e-41
1.490215036127766e-79
8.88061978550379e-37
2.3999575883825626e-32
2.419628779884569e-94
1.140563811641789e-73
4.5212248415444474e-69
3.045768524561105e-64
1.657471530406451e-36
3.816803194870377e-113
2.64046389660411e-80
1.2200239041008703e-53
1.7946046479309928e-100
5.156216286084331e-125
6.092442638838887e-44
1.177807726653794e-184
2.7738654392718286e-48
2.600842027427191e-88
8.525743657920824e-121
1.2300498951456379e-98
2.5471070137871858e-43
2.031003151532707e-58
6.980604908938156e-103
4.5948460465931294e-104
2.6207372224180104e-161
4.5264523715932197e-147
2.067884892876408e-78
7.470482748008943e-80
1.7480553313169e-163
1.78806039621921e-77
2.433085653058407e-56
6.767278264387451e-109
1.9365

todo

now sampled points along the trajectory from a starting point to an end point. 
trajectory using linear interpolation

now, concatenate a sample using this trajectory and the concat synthesizer

In [102]:
output_indices
# df[]

[np.int64(7648),
 np.int64(1780),
 np.int64(9064),
 np.int64(9099),
 np.int64(3051),
 np.int64(8371),
 np.int64(8627),
 np.int64(5819),
 np.int64(8638),
 np.int64(2131),
 np.int64(3478),
 np.int64(3264),
 np.int64(3317),
 np.int64(8596),
 np.int64(5837),
 np.int64(1),
 np.int64(6634),
 np.int64(3295),
 np.int64(5258),
 np.int64(3295),
 np.int64(3295),
 np.int64(3428),
 np.int64(3428),
 np.int64(3396),
 np.int64(3295),
 np.int64(3615),
 np.int64(5848),
 np.int64(3282),
 np.int64(3545),
 np.int64(3550),
 np.int64(3297),
 np.int64(5848),
 np.int64(3675),
 np.int64(3490),
 np.int64(3965),
 np.int64(6634),
 np.int64(6634),
 np.int64(3675),
 np.int64(3396),
 np.int64(3675),
 np.int64(5822),
 np.int64(3118),
 np.int64(8331),
 np.int64(3298),
 np.int64(8584),
 np.int64(8584),
 np.int64(5849),
 np.int64(5851),
 np.int64(5853),
 np.int64(5256)]